# Metadynamics Theory and Practice

Welcome to the dedicated environment for **Metadynamics (MTD)** in the `MTD` folder.

## 1. What is Metadynamics?
Metadynamics is an enhanced sampling technique used in Molecular Dynamics (MD) to explore the free energy surface (FES) of a system by adding a history-dependent bias potential along a set of **Collective Variables (CVs)**.

### Well-tempered Metadynamics (WTMetaD)
Currently, the most common variant is **Well-tempered Metadynamics**, where the height of the added Gaussians decreases over time to ensure convergence.

$$\Delta V(s, t) = W \exp \left( -\frac{|s - s(t)|^2}{2 \sigma^2} \right) e^{-V(s, t)/k_B \Delta T}$$

## 2. Working with PLUMED Data in Python
PLUMED typically outputs data in column-based text files (`COLVAR`, `HILLS`). Since the `plumed` Python wrapper on Windows requires a specialized build, we use `pandas` to efficiently load and analyze our data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", context="notebook")

print("Libraries loaded successfully.")

## 3. Loading a COLVAR File
PLUMED files have a header starting with `#! FIELDS`. We can use `pandas` to skip the metadata and load the data.

In [ ]:
def load_colvar(filename):
    """Helper to load PLUMED COLVAR files skipping the header."""
    # Get column names from the second line (fields)
    with open(filename, 'r') as f:
        line = f.readline()
        if line.startswith('#! FIELDS'):
            cols = line.split()[2:]
        else:
            # Try second line
            line = f.readline()
            cols = line.split()[2:]
            
    # Load data using pandas
    data = pd.read_csv(filename, sep='\s+', comment='#', names=cols)
    return data

print("Metadynamics helper functions defined.")

## 4. Visualizing Trajectories in CV Space
Tracking the evolution of CVs over time helps verify if the system has explored the relevant phase space.

In [ ]:
# Example of plotting (requires actual COLVAR file)
def plot_cv_evolution(data, cv_name='cv'):
    plt.figure(figsize=(10, 5))
    plt.plot(data['time'], data[cv_name], alpha=0.7)
    plt.xlabel('Time (ps)')
    plt.ylabel(cv_name)
    plt.title(f'Evolution of {cv_name} over time')
    plt.show()

## 5. Free Energy Surface (FES)
The FES can be reconstructed from the `sum_hills` tool in PLUMED. The resulting `fes.dat` can be visualized in Python as a contour plot or heat map.

### Typical Analysis Workflow:
1. Run LAMMPS + PLUMED.
2. Check convergence of hills.
3. Use `plumed sum_hills` to integrate bias potential into FES.
4. Plot FES in Python to identify minima and barriers.